In [2]:
import re
import numpy as np
import tensorflow as tf
import os

os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.text import Tokenizer

import kagglehub

c:\Users\karki\anaconda3\envs\keras\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Data

In [4]:
path = kagglehub.dataset_download("rohitgr/wikitext")
wikitext2_path = os.path.join(path, "wikitext-2")

def load_file(base_path, filename):
    with open(os.path.join(base_path, filename), "r", encoding="utf-8") as f:
        return f.read()

train_text = load_file(wikitext2_path, "wiki.train.tokens")
val_text = load_file(wikitext2_path, "wiki.valid.tokens")

## 2. Data Cleaning

In [ ]:
def clean_text(text):
    text = text.lower()

    text = re.sub(r"=+.*?=+", "", text)

    text = re.sub(r"[^a-zA-Z0-9\s\.,']", "", text)

    text = re.sub(r"\s+", " ", text)
    return text

train_text = clean_text(train_text)
val_text = clean_text(val_text)

train_text = train_text[:2_000_000]
val_text = val_text[:100_000]

## 2. Tokenizer

In [6]:
vocab_size_limit = 30000

tokenizer = Tokenizer(
    num_words=vocab_size_limit,
    oov_token="<OOV>",
    filters=""
)

tokenizer.fit_on_texts([train_text])

sequence = tokenizer.texts_to_sequences([train_text])[0]

word_index = tokenizer.word_index
vocab_size = min(vocab_size_limit, len(word_index) + 1)

index_to_word = {i: w for w, i in word_index.items()}
index_to_word[0] = "<PAD>"
index_to_word[word_index["<OOV>"]] = "<OOV>"

seq_length = 64

## 3. Dataset

In [ ]:
X, y = [], []

for i in range(0, len(sequence) - seq_length, 2):
    X.append(sequence[i:i+seq_length])
    y.append(sequence[i+1:i+seq_length+1])

X = np.array(X)
y = np.array(y)

idx = np.random.permutation(len(X))
X = X[idx]
y = y[idx]

split = int(0.9 * len(X))

X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]

## 4. Callback

In [8]:
callback = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

## 6. Sampling utility

In [9]:
def sample_probs(preds, temperature=1.0):
    preds = np.asarray(preds).astype("float64")
    preds = np.log(preds + 1e-8) / temperature
    preds = preds - np.max(preds)
    exp_preds = np.exp(preds)
    return exp_preds / np.sum(exp_preds)

## 7. Transformer Model

In [10]:
class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()

        self.att = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads,
            dropout=0.2
        )

        self.ffn = keras.Sequential([
            layers.Dense(ff_dim, activation="gelu"),
            layers.Dropout(0.2),
            layers.Dense(embed_dim),
        ])

        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)

        self.dropout = layers.Dropout(0.2)

    def call(self, inputs, training=False):
        seq_len = tf.shape(inputs)[1]

        causal_mask = tf.linalg.band_part(
            tf.ones((seq_len, seq_len)), -1, 0
        )

        attn = self.att(inputs, inputs, attention_mask=causal_mask)
        attn = self.dropout(attn, training=training)

        x = self.norm1(inputs + attn)

        ffn = self.ffn(x)
        ffn = self.dropout(ffn, training=training)

        return self.norm2(x + ffn)


class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim):
        super().__init__()

        self.token_emb = layers.Embedding(vocab_size, embed_dim)
        self.pos_emb = layers.Embedding(maxlen, embed_dim)
        self.dropout = layers.Dropout(0.2)

    def call(self, x):
        seq_len = tf.shape(x)[1]
        positions = tf.range(seq_len)
        x = self.token_emb(x)
        pos = self.pos_emb(positions)
        return self.dropout(x + pos)


embed_dim = 256
num_heads = 8
ff_dim = 512

inputs = layers.Input(shape=(seq_length,))
x = TokenAndPositionEmbedding(seq_length, vocab_size, embed_dim)(inputs)

x = TransformerBlock(embed_dim, num_heads, ff_dim)(x)
x = TransformerBlock(embed_dim, num_heads, ff_dim)(x)
x = TransformerBlock(embed_dim, num_heads, ff_dim)(x)
x = TransformerBlock(embed_dim, num_heads, ff_dim)(x)

x = layers.Dense(256, activation="gelu")(x)
x = layers.Dropout(0.2)(x)

outputs = layers.Dense(vocab_size, activation="softmax")(x)

transformer_model = keras.Model(inputs, outputs)

transformer_model.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(),
    optimizer=keras.optimizers.Adam(learning_rate=3e-4, clipnorm=1.0),
    metrics=["accuracy"]
)

## 8. Train

In [11]:
transformer_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=64,
    callbacks=[callback]
)

Epoch 1/20
2636/2636 ━━━━━━━━━━━━━━━━━━━━ 1566s 592ms/step - accuracy: 0.2070 - loss: 5.0539 - val_accuracy: 0.3230 - val_loss: 3.4419
Epoch 2/20
2636/2636 ━━━━━━━━━━━━━━━━━━━━ 1520s 577ms/step - accuracy: 0.3425 - loss: 3.2400 - val_accuracy: 0.5227 - val_loss: 2.0907
Epoch 3/20
2636/2636 ━━━━━━━━━━━━━━━━━━━━ 1437s 545ms/step - accuracy: 0.4439 - loss: 2.4864 - val_accuracy: 0.6502 - val_loss: 1.4616
Epoch 4/20
2636/2636 ━━━━━━━━━━━━━━━━━━━━ 1412s 536ms/step - accuracy: 0.5077 - loss: 2.0959 - val_accuracy: 0.7267 - val_loss: 1.1275
Epoch 5/20
2636/2636 ━━━━━━━━━━━━━━━━━━━━ 1444s 548ms/step - accuracy: 0.5510 - loss: 1.8577 - val_accuracy: 0.7756 - val_loss: 0.9237
Epoch 6/20
2636/2636 ━━━━━━━━━━━━━━━━━━━━ 1515s 575ms/step - accuracy: 0.5823 - loss: 1.6964 - val_accuracy: 0.8124 - val_loss: 0.7901
Epoch 7/20
2636/2636 ━━━━━━━━━━━━━━━━━━━━ 1486s 564ms/step - accuracy: 0.6065 - loss: 1.5801 - val_accuracy: 0.8359 - val_loss: 0.6956
Epoch 8/20
2636/2636 ━━━━━━━━━━━━━━━━━━━━ 1501s 569ms/s

# SAVE

In [ ]:
""" 
transformer_model.save("best_model.keras")

import pickle

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

with open("index_to_word.pkl", "wb") as f:
    pickle.dump(index_to_word, f)

config = {
    "seq_length": seq_length
}

with open("config.pkl", "wb") as f:
    pickle.dump(config, f)
 """


# LOAD

In [ ]:
"""
import pickle
from tensorflow import keras
from tensorflow.keras import layers
import tensorflow as tf

class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, **kwargs):
        super().__init__(**kwargs)

        self.att = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads,
            dropout=0.2
        )

        self.ffn = keras.Sequential([
            layers.Dense(ff_dim, activation="gelu"),
            layers.Dropout(0.2),
            layers.Dense(embed_dim),
        ])

        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout = layers.Dropout(0.2)

    def call(self, inputs, training=False):
        seq_len = tf.shape(inputs)[1]

        causal_mask = tf.linalg.band_part(
            tf.ones((seq_len, seq_len)), -1, 0
        )

        attn = self.att(inputs, inputs, attention_mask=causal_mask)
        attn = self.dropout(attn, training=training)

        x = self.norm1(inputs + attn)

        ffn = self.ffn(x)
        ffn = self.dropout(ffn, training=training)

        return self.norm2(x + ffn)


class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim, **kwargs):
        super().__init__(**kwargs)

        self.token_emb = layers.Embedding(vocab_size, embed_dim)
        self.pos_emb = layers.Embedding(maxlen, embed_dim)
        self.dropout = layers.Dropout(0.2)

    def call(self, x):
        seq_len = tf.shape(x)[1]
        positions = tf.range(seq_len)

        x = self.token_emb(x)
        pos = self.pos_emb(positions)

        return self.dropout(x + pos)
transformer_model = tf.keras.models.load_model(
    "best_model.keras",
    custom_objects={
        "TransformerBlock": TransformerBlock,
        "TokenAndPositionEmbedding": TokenAndPositionEmbedding
    }
)

with open("tokenizer.pkl", "rb") as f:
    tokenizer = pickle.load(f)

with open("index_to_word.pkl", "rb") as f:
    index_to_word = pickle.load(f)

with open("config.pkl", "rb") as f:
    config = pickle.load(f)

seq_length = config["seq_length"]

def predict_top_k(text, model, k=10, temperature=1.0):
    seq = tokenizer.texts_to_sequences([text])[0]
    seq = seq[-seq_length:]

    if len(seq) < seq_length:
        seq = [0] * (seq_length - len(seq)) + seq

    preds = model.predict(np.array([seq]), verbose=0)[0][-1]
    preds = np.log(preds + 1e-8) / temperature
    preds = np.exp(preds)
    preds = preds / np.sum(preds)

    top_k_indices = np.argsort(preds)[::-1]

    results = []
    for idx in top_k_indices:
        word = index_to_word.get(idx, "<UNK>")

        if word in ["<PAD>", "<OOV>", "unk", ".", ",", "'"]:
            continue

        results.append((word, float(preds[idx])))

        if len(results) >= k:
            break

    return results

def generate_text(seed_text, model, num_words=20, temperature=0.8):
    text = seed_text
    used_words = []

    for _ in range(num_words):
        probs = predict_top_k(text, model, k=len(index_to_word), temperature=temperature)

        words, probs = zip(*probs)
        probs = np.array(probs)

        for i, w in enumerate(words):
            if w in used_words[-5:]:
                probs[i] *= 0.3

        probs = probs / np.sum(probs)

        next_word = np.random.choice(words, p=probs)

        used_words.append(next_word)
        text += " " + next_word

    return text

print("\n10 predictions:")
for w, p in predict_top_k("the meaning of life", transformer_model):
    print(f"{w}: {p:.4f}")

print("\nGenerated text:")
print(generate_text("the meaning of life", transformer_model))
"""


c:\Users\karki\anaconda3\envs\keras\Lib\site-packages\keras\src\layers\layer.py:424: UserWarning: `build()` was called on layer 'token_and_position_embedding_1', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
c:\Users\karki\anaconda3\envs\keras\Lib\site-packages\keras\src\layers\layer.py:424: UserWarning: `build()` was called on layer 'transformer_block_4', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
c:\Users\karki\anaconda3\envs\keras\Lib\site-packages\keras\src\layers\layer.py:424: UserWarning: `bu


10 predictions:
as: 0.3761
and: 0.1111
but: 0.0377
in: 0.0353
giving: 0.0210
with: 0.0206
or: 0.0129
from: 0.0116
to: 0.0057
it: 0.0045

Generated text:
the meaning of life as a compounds in the first half of a highly brittle and crystalline metal pop metal and has been fashioned


# Text Generation testing

In [ ]:

def sample_top_p(probs, p=0.9):
    probs = np.asarray(probs).astype("float64")

    sorted_idx = np.argsort(probs)[::-1]
    sorted_probs = probs[sorted_idx]

    cum_probs = np.cumsum(sorted_probs)

    cutoff = cum_probs > p
    cutoff[1:] = cutoff[:-1].copy()
    cutoff[0] = False

    sorted_probs = sorted_probs[~cutoff]
    sorted_idx = sorted_idx[~cutoff]

    sorted_probs = sorted_probs / np.sum(sorted_probs)

    return np.random.choice(sorted_idx, p=sorted_probs)

def apply_temperature(probs, temperature=0.7):
    probs = np.asarray(probs).astype("float64")
    probs = np.log(probs + 1e-8) / temperature
    probs = np.exp(probs)
    return probs / np.sum(probs)

def predict_top_k(text, model, k=10, temperature=0.7):
    seq = tokenizer.texts_to_sequences([text])[0]
    seq = seq[-seq_length:]

    if len(seq) < seq_length:
        seq = [0] * (seq_length - len(seq)) + seq

    preds = model.predict(np.array([seq]), verbose=0)[0]
    preds = preds[-1]

    preds = apply_temperature(preds, temperature)

    top_k_idx = np.argsort(preds)[::-1]

    results = []
    for i in top_k_idx:
        word = index_to_word.get(i, "<UNK>")

        if word in ["<PAD>", "<OOV>", "unk", ".", ",", "'"]:
            continue

        results.append((word, float(preds[i])))

        if len(results) >= k:
            break

    return results

def generate_text(seed_text, model, num_words=30):
    text = seed_text
    seen_words = set()

    for _ in range(num_words):

        preds = predict_top_k(
            text,
            model,
            k=50,
            temperature=0.7
        )

        words, probs = zip(*preds)
        probs = np.array(probs)

        probs = probs / np.sum(probs)

        filtered_words = []
        filtered_probs = []

        for w, p in zip(words, probs):
            if w not in seen_words:
                filtered_words.append(w)
                filtered_probs.append(p)

        if len(filtered_words) == 0:
            filtered_words, filtered_probs = words, probs

        filtered_probs = np.array(filtered_probs)
        filtered_probs = filtered_probs / np.sum(filtered_probs)

        next_word = np.random.choice(filtered_words, p=filtered_probs)

        text += " " + next_word
        seen_words.add(next_word)

    return text

print("\n10 predictions:")
for w, p in predict_top_k("Artificial intelligence is used in", transformer_model):
    print(f"{w}: {p:.4f}")

print("\nGenerated text:")
print(generate_text("Artificial intelligence is used in", transformer_model))


10 predictions:
the: 0.5805
a: 0.3220
all: 0.0010
such: 0.0009
1939: 0.0005
general: 0.0003
order: 0.0003
1996: 0.0002
battle: 0.0001
development: 0.0001

Generated text:
Artificial intelligence is used in a compounds such as the first half of 2 and 1 to be carried out in some cases by aml will also have been made for more than predicted or


## 9. Text Generation

In [22]:
def predict_top_k(text, model, k=10):
    seq = tokenizer.texts_to_sequences([text])[0]
    seq = seq[-seq_length:]

    if len(seq) < seq_length:
        seq = [0] * (seq_length - len(seq)) + seq

    preds = model.predict(np.array([seq]), verbose=0)[0][-1]

    top_k_indices = np.argsort(preds)[::-1]

    results = []
    for idx in top_k_indices:
        word = index_to_word.get(idx, "<UNK>")

        if word in ["<PAD>", "<OOV>", "unk", ".", ",", "'"]:
            continue

        results.append((word, float(preds[idx])))

        if len(results) >= k:
            break

    return results

In [23]:
def generate_text(seed_text, model, num_words=20, temperature=0.8):
    text = seed_text
    used_words = []

    for _ in range(num_words):
        probs = predict_top_k(text, model, k=len(index_to_word), temperature=temperature)

        words, probs = zip(*probs)
        probs = np.array(probs)

        for i, w in enumerate(words):
            if w in used_words[-5:]:
                probs[i] *= 0.3

        probs = probs / np.sum(probs)

        next_word = np.random.choice(words, p=probs)

        used_words.append(next_word)
        text += " " + next_word

    return text

In [ ]:
print("\n10 predictions:")
for w, p in predict_top_k("Artificial intelligence is used in", transformer_model):
    print(f"{w}: {p:.4f}")

print("\nGenerated text:")
print(generate_text("Artificial intelligence is used in", transformer_model))



10 predictions:
as: 0.5225
and: 0.0915
but: 0.0196
in: 0.0178
giving: 0.0085
with: 0.0083
or: 0.0042
from: 0.0036
to: 0.0013
it: 0.0009

Generated text:
Artificial intelligence is used in the first time to be a site in ireland and worse than galleys they replaced by one new zealand for conservation of which were more difficult or two stallions when
